# Voz de Gabriela en Piper

Afina [Piper](https://github.com/OHF-Voice/piper1-gpl) con el dataset que genera
`pipeline/dataset_piper.py` (F5-TTS como maestra). Sale un `.onnx` de ~60 MB
que sintetiza más rápido que tiempo real en CPU.

**Antes de empezar**

1. *Entorno de ejecución → Cambiar tipo → GPU T4*.
2. En el Mac, empaqueta el dataset:
   ```bash
   cd assets/voz/dataset && zip -rq ../dataset-piper.zip metadata.csv wav
   ```
3. Sube `dataset-piper.zip` a Google Drive, en `Mi unidad/gabriela-piper/`.

Colab gratis corta la sesión a las pocas horas. No pasa nada: los checkpoints
quedan en Drive y al volver a correr todo, el entrenamiento **sigue donde
quedó**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
DRIVE = Path("/content/drive/MyDrive/gabriela-piper")
ZIP = DRIVE / "dataset-piper.zip"
ENTRENO = DRIVE / "entreno"  # checkpoints: sobreviven a que Colab corte
ENTRENO.mkdir(parents=True, exist_ok=True)
assert ZIP.exists(), f"falta {ZIP}: súbelo a Drive primero"

## Instalar Piper

Fijado a un commit concreto: `main` cambia y el entrenamiento no debería
cambiar con él. Unos 5 minutos.

In [ ]:
PIPER_COMMIT = "5b355b110aecf3de8f4e000ede1ce06831acff35"  # main, 17-09-2026
!apt-get -qq install -y build-essential cmake ninja-build > /dev/null
!git clone -q https://github.com/OHF-voice/piper1-gpl.git /content/piper1-gpl
%cd /content/piper1-gpl
!git checkout -q {PIPER_COMMIT}
!pip install -q -e '.[train]'
!./build_monotonic_align.sh
!python3 setup.py build_ext --inplace > /dev/null

## Datos y checkpoint base

Se parte de `es_MX/ald/medium`: español latinoamericano, una sola voz, dataset
con licencia Unlicense. Afinar desde ahí es mucho más rápido que desde cero,
aunque la voz de partida sea otra: el timbre se reemplaza, la pronunciación se
conserva.

In [ ]:
import zipfile
DATOS = Path("/content/dataset")
if not DATOS.exists():
    zipfile.ZipFile(ZIP).extractall(DATOS)
print(sum(1 for _ in open(DATOS / "metadata.csv")), "frases")

from huggingface_hub import hf_hub_download
BASE = hf_hub_download("rhasspy/piper-checkpoints",
                       "es/es_MX/ald/medium/epoch=9999-step=1753600.ckpt",
                       repo_type="dataset")

## Entrenar

Corre sin fin: páralo con ■ cuando suene bien, o deja que Colab lo corte.
Cada validación deja `last.ckpt` en Drive, y la próxima vez se retoma desde el
más reciente.

Sólo se guardan dos checkpoints —el último y el de menor `val_mel`—, de ~850 MB
cada uno. Los de Piper por defecto son once y no caben en los 15 GB gratis de
Drive.

Como referencia, las voces de Piper afinadas suelen necesitar del orden de mil
épocas más sobre la base. Oye el resultado antes de dar por buena una cifra.

In [ ]:
CONFIG = ENTRENO / "checkpoints.yaml"
CONFIG.write_text('''
trainer:
  callbacks:
    - class_path: lightning.pytorch.callbacks.ModelCheckpoint
      init_args:
        monitor: val_mel
        mode: min
        save_top_k: 1
        save_last: true
''')

previos = sorted(ENTRENO.glob("lightning_logs/*/checkpoints/last.ckpt"),
                 key=lambda p: p.stat().st_mtime)
DESDE = previos[-1] if previos else BASE
print("parte desde", DESDE)

!python3 -m piper.train fit \
  --config "{CONFIG}" \
  --trainer.default_root_dir "{ENTRENO}" \
  --data.voice_name gabriela \
  --data.csv_path "{DATOS}/metadata.csv" \
  --data.audio_dir "{DATOS}/wav" \
  --model.sample_rate 22050 \
  --data.espeak_voice es-419 \
  --data.cache_dir /content/cache \
  --data.config_path "{ENTRENO}/config.json" \
  --data.batch_size 32 \
  --ckpt_path "{DESDE}"

## Exportar y oír

Exporta el último checkpoint a ONNX y sintetiza una frase **en CPU**, que es
como correría en el clúster. El factor de tiempo real que imprime es el número
que importa: bajo 1 es más rápido que hablar.

In [ ]:
ultimo = sorted(ENTRENO.glob("lightning_logs/*/checkpoints/last.ckpt"),
                key=lambda p: p.stat().st_mtime)[-1]
ONNX = DRIVE / "es_419-gabriela-medium.onnx"
!python3 -m piper.train.export_onnx --checkpoint "{ultimo}" --output-file "{ONNX}"
!cp "{ENTRENO}/config.json" "{ONNX}.json"
print("exportado", ultimo.parent.parent.name, "→", ONNX)

In [ ]:
import time, wave
from IPython.display import Audio
from piper import PiperVoice

voz = PiperVoice.load(str(ONNX))  # CPU por defecto
TEXTO = ("Fue en esta casa donde rendí mis exámenes de habilitación, "
         "el año diez. No tenía título de la Escuela Normal.")
t0 = time.time()
with wave.open("/content/prueba.wav", "wb") as w:
    voz.synthesize_wav(TEXTO, w)
tarda = time.time() - t0
with wave.open("/content/prueba.wav") as w:
    dura = w.getnframes() / w.getframerate()
print(f"{dura:.1f} s de audio en {tarda:.2f} s: factor de tiempo real {tarda / dura:.2f}")
Audio("/content/prueba.wav")